# 🎓 Chapter 10-03a: GenAI Interview Mastery

---

## 📖 The Final Boss

You've learned prompting, chains, RAG, agents, LangGraph, multi-agent systems, fine-tuning, and deployment. Now comes the **interview**.

This notebook is your **ultimate revision guide** — 50+ questions covering every chapter, organized by difficulty.

---

## 🧠 Chapter 1: Foundations

<details>
<summary><strong>Q1: What is Generative AI vs Traditional ML?</strong></summary>

**Traditional ML**: Maps input → output (classification, regression). Learns patterns FROM data.
**Generative AI**: Creates NEW content (text, images, code). Uses neural networks to learn data distributions and sample from them. Key architectures: Transformers (GPT, BERT), Diffusion Models (DALL-E, Stable Diffusion), GANs.

</details>

<details>
<summary><strong>Q2: Explain the Transformer architecture in simple terms.</strong></summary>

**Self-Attention**: Each token looks at ALL other tokens to understand context ("bank" near "river" vs "money").
**Key components**: (1) Token + Positional Embeddings, (2) Multi-Head Self-Attention, (3) Feed-Forward Networks, (4) Layer Normalization.
**Encoder-only**: BERT (understanding). **Decoder-only**: GPT (generation). **Encoder-Decoder**: T5 (translation).

</details>

<details>
<summary><strong>Q3: What are tokens? Why do they matter?</strong></summary>

Tokens are sub-word units (BPE — Byte Pair Encoding). "unhappiness" → ["un", "happiness"]. They matter because: (1) LLMs have **context windows** measured in tokens (4K-128K), (2) **Pricing** is per token, (3) **Latency** scales with token count. Roughly 1 token ≈ 4 characters in English.

</details>

<details>
<summary><strong>Q4: What's the difference between temperature and top_p?</strong></summary>

Both control randomness. **Temperature**: Scales logits before softmax. T=0 → deterministic, T=1 → default, T>1 → more random. **Top-p (nucleus sampling)**: Only considers tokens whose cumulative probability ≤ p. Top_p=0.1 → picks from top 10% probability mass. Use temperature for creativity control, top_p for diversity.

</details>

<details>
<summary><strong>Q5: What are embeddings and why are they important?</strong></summary>

Dense vector representations of text in high-dimensional space (768-3072 dims). Similar meanings → similar vectors. Critical for: (1) **Semantic search** — find relevant docs by meaning, (2) **RAG** — retrieve context for LLMs, (3) **Clustering** — group similar content. Models: OpenAI text-embedding-3-small, Sentence-Transformers.

</details>

---

## 🧠 Chapter 2: Prompt Engineering

<details>
<summary><strong>Q6: What are the key prompt engineering techniques?</strong></summary>

1. **Zero-shot**: Direct question, no examples
2. **Few-shot**: Provide 2-5 examples before the question
3. **Chain-of-Thought (CoT)**: "Let's think step by step"
4. **ReAct**: Thought → Action → Observation loop
5. **Tree-of-Thought (ToT)**: Explore multiple reasoning paths
6. **Self-Consistency**: Multiple CoT paths → majority vote

</details>

<details>
<summary><strong>Q7: When would you use Chain-of-Thought vs Few-Shot?</strong></summary>

**Few-shot**: When you need consistent FORMAT ("always output JSON like this"). Works for classification, extraction, formatting.
**CoT**: When you need REASONING (math, logic, multi-step analysis). The model shows its work, reducing errors on complex tasks. Combine both: few-shot with CoT examples for best results.

</details>

<details>
<summary><strong>Q8: What is structured output and why use it?</strong></summary>

Force LLM to output valid JSON/XML matching a schema. Methods: (1) **Pydantic + with_structured_output()** — LangChain native, (2) **OpenAI JSON mode** — response_format={"type": "json_object"}, (3) **Function calling** — tool schemas enforce structure. Use when downstream code needs to parse the output reliably.

</details>

---

## 🧠 Chapter 3: LangChain Core

<details>
<summary><strong>Q9: What is LCEL and why does it matter?</strong></summary>

**LangChain Expression Language**: Composable pipe syntax for building chains. `prompt | llm | parser`. Benefits: (1) Automatic streaming, (2) Async support, (3) Batch processing, (4) Tracing with LangSmith. Replaced legacy LLMChain/SequentialChain in LangChain v0.3+.

</details>

<details>
<summary><strong>Q10: Explain RunnablePassthrough, RunnableLambda, and RunnableParallel.</strong></summary>

- **RunnablePassthrough**: Passes input through unchanged. Used with `.assign()` to add computed fields while keeping original data
- **RunnableLambda**: Wraps any Python function as a Runnable. `RunnableLambda(lambda x: x.upper())`
- **RunnableParallel**: Runs multiple chains simultaneously. `{"summary": chain1, "keywords": chain2}` → both run in parallel

</details>

<details>
<summary><strong>Q11: How does tool calling work in LangChain?</strong></summary>

1. Define tool with `@tool` decorator or Pydantic schema
2. Bind to model: `llm.bind_tools([search_tool, calc_tool])`
3. Model returns `AIMessage` with `tool_calls` list (name + args)
4. Execute tool, return `ToolMessage` with result
5. Send back to model for final response

The model DOESN'T execute tools — it REQUESTS them. Your code runs them.

</details>

---

## 🧠 Chapter 4: Embeddings & Vector Stores

<details>
<summary><strong>Q12: How does cosine similarity work for semantic search?</strong></summary>

Measures angle between two vectors: `cos(θ) = (A·B) / (||A|| × ||B||)`. Range: -1 to 1 (1 = identical direction). Used because it's magnitude-invariant — a long document and short query can still match. Alternative: dot product (faster, used by FAISS), Euclidean distance (sensitive to magnitude).

</details>

<details>
<summary><strong>Q13: Compare ChromaDB, FAISS, and Pinecone.</strong></summary>

| Feature | ChromaDB | FAISS | Pinecone |
|---------|----------|-------|----------|
| Type | Embedded DB | Library | Managed Cloud |
| Scale | <1M docs | Billions | Unlimited |
| Metadata Filter | ✅ | ❌ | ✅ |
| Persistence | SQLite | File-based | Cloud |
| Best For | Prototyping | High-perf local | Production SaaS |

</details>

<details>
<summary><strong>Q14: What is HNSW and why is it used?</strong></summary>

**Hierarchical Navigable Small World** — an ANN (Approximate Nearest Neighbor) algorithm. Builds a multi-layer graph where top layers have long-range connections (coarse search) and bottom layers have short-range (fine search). O(log n) search vs O(n) brute force. Used by ChromaDB, Qdrant, Weaviate. Trade-off: small accuracy loss for massive speed gain.

</details>

---

## 🧠 Chapter 5: RAG

<details>
<summary><strong>Q15: Explain the RAG pipeline step by step.</strong></summary>

**Indexing** (offline): Load docs → Split into chunks → Embed chunks → Store in vector DB.
**Retrieval** (runtime): Embed query → Search vector DB → Get top-k chunks.
**Generation**: Inject chunks into prompt → LLM generates grounded answer.

Key decisions: chunk size (500-1000 chars), overlap (10-20%), embedding model, top-k (3-5).

</details>

<details>
<summary><strong>Q16: What is the "Lost in the Middle" problem?</strong></summary>

LLMs attend more to the BEGINNING and END of context, often ignoring information in the MIDDLE. Solution: (1) Keep retrieved chunks small and relevant, (2) Re-rank so most relevant is first, (3) Use MAP-Rerank to score each chunk independently, (4) Limit to 3-5 chunks rather than stuffing 20.

</details>

<details>
<summary><strong>Q17: What are advanced RAG techniques?</strong></summary>

1. **Multi-Query**: Rephrase question 3-5 ways → retrieve for each → merge results
2. **HyDE**: Generate hypothetical answer → embed THAT → search (better alignment)
3. **Hybrid Search**: Combine BM25 (keyword) + vector search with RRF fusion
4. **Re-ranking**: Use cross-encoder to re-score retrieved chunks
5. **CRAG**: LLM evaluates retrieval quality → web search fallback if poor
6. **Parent Document Retrieval**: Embed child chunks, return parent documents

</details>

<details>
<summary><strong>Q18: How do you evaluate a RAG system?</strong></summary>

**RAGAS Metrics**: (1) **Faithfulness** — is the answer grounded in retrieved context? (2) **Answer Relevance** — does it actually answer the question? (3) **Context Precision** — are the retrieved chunks relevant? (4) **Context Recall** — did we retrieve ALL needed info?

Also: LLM-as-Judge, human evaluation, A/B testing on user satisfaction.

</details>

---

## 🧠 Chapter 6: Agents

<details>
<summary><strong>Q19: What is the ReAct framework?</strong></summary>

**Reasoning + Acting**. Loop: (1) **Thought** — LLM reasons about what to do, (2) **Action** — calls a tool (search, calculate, etc.), (3) **Observation** — gets tool result, (4) Repeat until Final Answer. Advantages: transparent reasoning, tool use, self-correction. Limitation: can get stuck in loops.

</details>

<details>
<summary><strong>Q20: What's the difference between tool calling and function calling?</strong></summary>

They're effectively the same concept, rebranded. **Function calling** was OpenAI's original term. **Tool calling** is the broader, provider-agnostic term used by LangChain. The model outputs structured JSON requesting a function execution — it does NOT execute any code itself. Your application code must execute the tool and return results.

</details>

<details>
<summary><strong>Q21: How do agents handle errors?</strong></summary>

(1) **Retry with feedback** — pass error message back to LLM, let it try a different approach. (2) **Max iterations** — cap the loop (e.g., 10 steps) to prevent infinite loops. (3) **Fallback tools** — if primary tool fails, try alternative. (4) **Graceful degradation** — return partial answer with disclaimer.

</details>

---

## 🧠 Chapter 7: LangGraph

<details>
<summary><strong>Q22: What is LangGraph and why use it over LCEL chains?</strong></summary>

LangGraph = state machine framework for building agent workflows. LCEL is LINEAR (A→B→C). LangGraph supports CYCLES (loops), CONDITIONAL routing, and PERSISTENT state. Use when you need: agent loops, human-in-the-loop approvals, multi-step workflows with branching, or checkpointing.

</details>

<details>
<summary><strong>Q23: Explain State, Nodes, and Edges in LangGraph.</strong></summary>

- **State**: TypedDict that flows through the graph. Annotated fields define how updates merge (e.g., `Annotated[list, add_messages]` appends messages)
- **Nodes**: Python functions that receive state and return partial state updates. `def my_node(state: MyState) -> dict:`
- **Edges**: Connect nodes. Normal edges (A→B always), Conditional edges (route based on state), Entry/finish points

</details>

<details>
<summary><strong>Q24: How does checkpointing work in LangGraph?</strong></summary>

**MemorySaver** (dev) or **SqliteSaver/PostgresSaver** (prod) saves state after every node execution. Benefits: (1) Resume interrupted workflows, (2) Time-travel debugging (inspect any past state), (3) Human-in-the-loop — pause, edit state, resume. Enabled via `graph.compile(checkpointer=MemorySaver())`. Thread IDs scope conversations.

</details>

<details>
<summary><strong>Q25: What is Human-in-the-Loop (HITL) in LangGraph?</strong></summary>

Three patterns: (1) **interrupt_before** — pause BEFORE a node executes (approval gate), (2) **interrupt_after** — pause AFTER a node for review, (3) **State editing** — modify state mid-execution via `graph.update_state()`. Use for: confirming tool calls, reviewing generated content, injecting human feedback.

</details>

---

## 🧠 Chapter 8: Multi-Agent Systems

<details>
<summary><strong>Q26: What are the main multi-agent patterns?</strong></summary>

1. **Supervisor**: Central orchestrator routes tasks to specialist workers
2. **Hierarchical**: Tree of supervisors (manager → team leads → workers)
3. **Handoff/Swarm**: Agents pass control directly to each other
4. **Debate**: Agents argue opposing positions → judge picks best
5. **Pipeline**: Sequential flow (researcher → writer → editor)

</details>

<details>
<summary><strong>Q27: Compare CrewAI vs LangGraph for multi-agent systems.</strong></summary>

| Aspect | CrewAI | LangGraph |
|--------|--------|----------|
| Abstraction | High (Agent/Task/Crew) | Low (State/Node/Edge) |
| Learning Curve | Easy | Steep |
| Flexibility | Limited | Total control |
| Debugging | Opaque | Full state inspection |
| Best For | Quick prototypes | Production systems |

</details>

<details>
<summary><strong>Q28: How do you handle shared state in multi-agent systems?</strong></summary>

Three approaches: (1) **Shared state graph** — all agents read/write same TypedDict (LangGraph native), (2) **Message passing** — agents communicate via message lists, (3) **Blackboard pattern** — shared memory store that agents read from and write to. Key: define clear state schemas and prevent race conditions.

</details>

---

## 🧠 Chapter 9: Fine-Tuning & MCP

<details>
<summary><strong>Q29: When should you fine-tune vs use RAG vs prompt engineering?</strong></summary>

```
Need domain knowledge?     → RAG (add external data)
Need behavior change?      → Fine-tuning (teach new style/format)
Need task formatting?      → Prompt engineering (cheapest first)
Need both?                → RAG + fine-tuned model
```

**Decision flow**: Try prompt engineering first → if insufficient, add RAG → if style/behavior still wrong, fine-tune.

</details>

<details>
<summary><strong>Q30: What is LoRA and QLoRA?</strong></summary>

**LoRA** (Low-Rank Adaptation): Instead of updating ALL model weights, inject small trainable matrices (rank 8-64) into attention layers. Trains only 0.1-1% of parameters. Original weights frozen.
**QLoRA**: Same as LoRA but loads base model in 4-bit quantization. Enables fine-tuning 70B models on a single GPU. The adapter weights stay in full precision.

</details>

<details>
<summary><strong>Q31: What is MCP (Model Context Protocol)?</strong></summary>

Open standard by Anthropic for connecting AI models to external tools/data. Architecture: **Host** (Claude, VS Code) → **Client** (session manager) → **Server** (tool provider). Three primitives: (1) **Tools** — functions the model can call, (2) **Resources** — data the model can read, (3) **Prompts** — reusable templates. Transport: stdio (local) or HTTP+SSE (remote).

</details>

---

## 🧠 Chapter 10: Production & Safety

<details>
<summary><strong>Q32: How do you optimize LLM costs in production?</strong></summary>

(1) **Caching** — exact match (InMemoryCache) or semantic cache (embed queries, match similar). (2) **Model routing** — use GPT-4o-mini for simple tasks, GPT-4o for complex ones. (3) **Prompt optimization** — shorter prompts = fewer tokens = lower cost. (4) **Batch processing** — aggregate API calls. (5) **Fine-tune smaller models** to replace expensive large models.

</details>

<details>
<summary><strong>Q33: How do you handle prompt injection in production?</strong></summary>

Multi-layer defense: (1) **Input sanitization** — regex for known patterns, (2) **Instruction hierarchy** — system prompt > user prompt, (3) **Delimiter isolation** — wrap user input in tags, (4) **LLM-as-Judge** — classify input before main model, (5) **Canary tokens** — detect system prompt leakage, (6) **Output scanning** — check for PII leakage in response.

</details>

<details>
<summary><strong>Q34: What monitoring do you need for LLM apps?</strong></summary>

**LangSmith** or equivalent for: (1) Trace every LLM call (input/output/latency/cost), (2) Error rates and failure modes, (3) Token usage and cost tracking, (4) Retrieval quality metrics (for RAG), (5) User feedback collection, (6) Regression testing with datasets. Also: standard infra monitoring (uptime, p99 latency, error rates).

</details>

---

## 🧠 System Design Questions

<details>
<summary><strong>Q35: Design a customer support chatbot for an e-commerce company.</strong></summary>

**Architecture**: 
- RAG over product catalog + FAQ docs + order history
- LangGraph agent with tools: search_orders, track_shipping, process_return, escalate_to_human
- Intent classifier routes: product_question → RAG, order_issue → tool agent, complaint → human
- Guardrails: PII redaction, topic restriction (no competitor comparisons)
- Memory: conversation history per session + user profile context
- Monitoring: LangSmith traces, CSAT scoring, escalation rate tracking

</details>

<details>
<summary><strong>Q36: Design a document Q&A system for a law firm (10M+ pages).</strong></summary>

**Challenges**: Scale, accuracy is critical, citations required.
**Architecture**:
- **Ingestion**: Document parsing (PDF/DOCX) → hierarchical chunking (section → paragraph)
- **Retrieval**: Hybrid search (BM25 + vector) → cross-encoder re-ranking → top-5 chunks
- **Generation**: GPT-4o with structured output including citations
- **Evaluation**: Faithfulness scoring, mandatory source attribution
- **Vector DB**: Pinecone or Weaviate (scale + metadata filtering by case type, date, jurisdiction)
- **Guardrails**: No legal advice disclaimers, PII protection for client data

</details>

<details>
<summary><strong>Q37: Design a multi-agent research assistant.</strong></summary>

**LangGraph Architecture**:
- **Planner Agent**: Decomposes research question into sub-queries
- **Researcher Agent**: Searches web + internal docs per sub-query
- **Analyst Agent**: Synthesizes findings, identifies gaps
- **Writer Agent**: Produces structured report with citations
- **Reviewer Agent**: Fact-checks claims, scores quality
- **Cycle**: If quality < threshold, send back to researcher with gaps
- **State**: SharedResearchState with findings, sources, draft, review_score
- **Checkpointing**: Save after each agent for resume/debugging

</details>

---

## 🧠 Rapid Fire (Concept Pairs)

<details>
<summary><strong>Q38: RAG vs Fine-Tuning</strong></summary>

RAG adds external knowledge at runtime (no training). Fine-tuning changes model behavior/style (requires training data). RAG for dynamic data, fine-tuning for consistent behavior.
</details>

<details>
<summary><strong>Q39: LCEL vs LangGraph</strong></summary>

LCEL = linear chains (prompt | llm | parser). LangGraph = stateful graphs with cycles, conditionals, and persistence. Use LCEL for simple pipelines, LangGraph for agents and complex workflows.
</details>

<details>
<summary><strong>Q40: Semantic Search vs Keyword Search</strong></summary>

Keyword (BM25): Exact word matching, fast, no ML needed. Semantic (vector): Meaning-based, catches synonyms, needs embedding model. Best: Hybrid combining both with RRF fusion.
</details>

<details>
<summary><strong>Q41: Agents vs Chains</strong></summary>

Chains: Fixed sequence, predictable. Agents: Dynamic decision-making, tool selection, loops. Chains for known workflows, agents when the path depends on the input.
</details>

<details>
<summary><strong>Q42: Supervisor vs Swarm Pattern</strong></summary>

Supervisor: Centralized router, single point of control, easier to debug. Swarm: Decentralized handoffs between peers, more autonomous, harder to trace. Supervisor for most production systems, swarm for creative/exploratory tasks.
</details>

<details>
<summary><strong>Q43: LoRA vs Full Fine-Tuning</strong></summary>

Full: Updates all parameters, best quality, needs massive GPU (A100 cluster). LoRA: Updates 0.1% of params via low-rank adapters, 90% of quality, single GPU. QLoRA adds 4-bit quantization for even less memory.
</details>

<details>
<summary><strong>Q44: Streaming vs Batch LLM Calls</strong></summary>

Streaming: Token-by-token output, better UX (perceived faster), use for chat. Batch: All at once, better for processing pipelines, use for background tasks.
</details>

<details>
<summary><strong>Q45: ChromaDB vs Pinecone</strong></summary>

ChromaDB: Open-source, embedded, great for prototyping, <1M docs. Pinecone: Managed cloud, auto-scaling, metadata filtering, unlimited scale. ChromaDB for dev, Pinecone for production SaaS.
</details>

---

## 🧠 Behavioral / Scenario Questions

<details>
<summary><strong>Q46: Your RAG system returns wrong answers for 20% of queries. How do you debug?</strong></summary>

Systematic debugging: (1) **Check retrieval** — are the right chunks coming back? Log retrieved docs. (2) **Check chunking** — are chunks too small/large? Is relevant info split across chunks? (3) **Check embedding quality** — try different embedding model. (4) **Add re-ranking** — cross-encoder to filter noise. (5) **Evaluate with RAGAS** — identify if it's retrieval failure or generation failure. (6) **A/B test** — compare different configs on a test set.

</details>

<details>
<summary><strong>Q47: Your LLM app costs $5000/month. How do you reduce to $1000?</strong></summary>

(1) **Audit token usage** — find the biggest cost drivers. (2) **Add caching** — semantic cache for similar queries (50-70% hit rate possible). (3) **Model cascade** — GPT-4o-mini for 80% of queries, GPT-4o only when confidence is low. (4) **Shorten prompts** — remove unnecessary context. (5) **Batch processing** — aggregate where possible. (6) **Fine-tune smaller model** — replace GPT-4 with fine-tuned GPT-3.5 for specific tasks.

</details>

<details>
<summary><strong>Q48: Your agent gets stuck in an infinite loop. What do you do?</strong></summary>

(1) **Immediate**: Add `recursion_limit` in LangGraph config. (2) **Root cause**: Check conditional edges — is the routing logic correct? Is tool output being misinterpreted? (3) **Fix**: Add a step counter in state, add explicit exit conditions, improve the system prompt to guide better decisions. (4) **Prevent**: Use LangSmith to trace agent steps, set up alerts for long-running traces.

</details>

---

## 🎯 Interview Cheat Sheet

```
╔══════════════════════════════════════════════════════════════════╗
║              GENAI INTERVIEW QUICK REFERENCE                    ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║  PROMPTING: Few-shot → CoT → ReAct → Structured Output        ║
║  LANGCHAIN: LCEL pipes | model.bind_tools() | OutputParser     ║
║  RAG: Load → Split → Embed → Store → Retrieve → Generate      ║
║  ADVANCED RAG: Multi-query, HyDE, Hybrid Search, Re-ranking   ║
║  AGENTS: ReAct loop, tool calling, LangGraph state machines    ║
║  LANGGRAPH: State + Nodes + Edges + Checkpointing + HITL      ║
║  MULTI-AGENT: Supervisor > Hierarchical > Swarm patterns       ║
║  FINE-TUNING: Prompt eng → RAG → LoRA/QLoRA (decision tree)   ║
║  PRODUCTION: Cache → Model routing → Guardrails → Monitoring   ║
║  SAFETY: PII detection → Injection defense → Content filter    ║
║                                                                ║
╚══════════════════════════════════════════════════════════════════╝
```

---

## ✅ Key Takeaways

```
┌───────────────────────────────────────────────────────────────────┐
│  📌 Know the WHY behind each technique, not just the HOW.       │
├───────────────────────────────────────────────────────────────────┤
│  📌 System design = Architecture + Trade-offs + Scale + Safety.  │
├───────────────────────────────────────────────────────────────────┤
│  📌 Always mention: caching, monitoring, guardrails, cost.      │
├───────────────────────────────────────────────────────────────────┤
│  📌 Compare frameworks: know when to use what and why.          │
├───────────────────────────────────────────────────────────────────┤
│  📌 Practice explaining concepts in 30 seconds — interviewers   │
│     want clarity, not a lecture.                                 │
└───────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Take-Home Project Challenge!

---

### 🎓 You're Interview-Ready!
48 questions across all chapters. Review, practice, and crush that interview! 💪